# News Article Data Overview EDA

This notebook gives a compact overview of the news article data used in the electronics price pressure project.

Focus areas:

- Where the data is loaded from
- Time coverage and monthly article volume
- Source distribution
- Missing values and duplicate URLs
- Article length distribution and long-article review

The notebook is intentionally descriptive. It does not train models, export files, or save plots.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "config.yaml").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root with configs/config.yaml")

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

EDA_DIR = ROOT / "artifacts" / "eda"
ROOT

## 1. Load News Articles

Load priority:

1. MongoDB clean-news collections from `configs/config.yaml`
2. `artifacts/eda/news_eda_rows.csv`, if it exists
3. `ai_lab/upload_bundle_next_ready/news_to_enrich.jsonl`, if available

This makes MongoDB the primary data source while keeping the notebook runnable in offline review mode.

In [ ]:
from src.storage import load_dataframe_from_mongo
from src.utils import load_config

config = load_config(ROOT / "configs" / "config.yaml")
mongo_cfg = config["storage"]["mongo"]

eda_rows_path = EDA_DIR / "news_eda_rows.csv"
monthly_artifact_path = EDA_DIR / "news_eda_monthly.csv"
ai_bundle_path = ROOT / "ai_lab" / "upload_bundle_next_ready" / "news_to_enrich.jsonl"

def load_from_mongo(config):
    candidate_collections = [
        mongo_cfg.get("test_clean_news_collection"),
        mongo_cfg.get("clean_news_collection"),
    ]
    candidate_collections = [name for name in candidate_collections if name]

    notes = []
    for collection_name in candidate_collections:
        try:
            frame = load_dataframe_from_mongo(config, collection_name, sort_by="published_at")
        except Exception as exc:
            notes.append(f"{collection_name}: {exc}")
            continue

        if not frame.empty:
            return frame, f"MongoDB collection: {collection_name}", notes

        notes.append(f"{collection_name}: empty collection")

    return pd.DataFrame(), None, notes

articles, data_source, load_notes = load_from_mongo(config)

if articles.empty and eda_rows_path.exists() and eda_rows_path.stat().st_size > 0:
    articles = pd.read_csv(eda_rows_path)
    data_source = str(eda_rows_path.relative_to(ROOT))

if articles.empty and ai_bundle_path.exists() and ai_bundle_path.stat().st_size > 0:
    articles = pd.read_json(ai_bundle_path, lines=True)
    data_source = str(ai_bundle_path.relative_to(ROOT))

if articles.empty:
    raise FileNotFoundError("No article data found in MongoDB, EDA CSV, or AI-LAB article bundle")

monthly_artifact = pd.read_csv(monthly_artifact_path, parse_dates=["month"]) if monthly_artifact_path.exists() else pd.DataFrame()

print(f"Data source used: {data_source}")
print(f"Loaded rows: {len(articles):,}")
if load_notes and not str(data_source).startswith("MongoDB"):
    print("MongoDB notes:")
    for note in load_notes:
        print(f"- {note}")

display(articles.head())

## 2. Standardize Core Fields

The pipeline can produce slightly different schemas depending on whether the data comes from MongoDB, an EDA artifact, or the AI-LAB bundle. This section standardizes the fields needed for overview EDA.

In [ ]:
df = articles.copy()

if "published_at" not in df.columns and "seen_date" in df.columns:
    df["published_at"] = df["seen_date"]

df["published_at"] = pd.to_datetime(df.get("published_at"), errors="coerce")
df = df.dropna(subset=["published_at"]).reset_index(drop=True)
df["month"] = df["published_at"].dt.to_period("M").dt.to_timestamp()
df["year"] = df["published_at"].dt.year

for col in ["title", "source", "url", "language", "source_country"]:
    if col not in df.columns:
        df[col] = ""

text_candidates = ["analysis_text", "clean_text", "text", "content", "title"]
available_text_cols = [col for col in text_candidates if col in df.columns]
df["eda_text"] = ""
for col in available_text_cols:
    values = df[col].fillna("").astype(str)
    df["eda_text"] = df["eda_text"].where(df["eda_text"].str.strip().ne(""), values)

if "article_word_count" in df.columns:
    df["word_count"] = pd.to_numeric(df["article_word_count"], errors="coerce")
else:
    df["word_count"] = df["eda_text"].fillna("").astype(str).str.split().str.len()

if "article_char_count" in df.columns:
    df["char_count"] = pd.to_numeric(df["article_char_count"], errors="coerce")
elif "content_char_count" in df.columns:
    df["char_count"] = pd.to_numeric(df["content_char_count"], errors="coerce")
else:
    df["char_count"] = df["eda_text"].fillna("").astype(str).str.len()

# A rough estimate that is useful when text is truncated but source character count is retained.
df["estimated_words"] = df["char_count"] / 5

display(df[["published_at", "source", "title", "word_count", "char_count", "estimated_words"]].head())

## 3. Dataset Shape and Schema

This gives a basic overview of the loaded data before looking at trends.

In [ ]:
schema_summary = pd.DataFrame(
    {
        "column": df.columns,
        "dtype": [str(df[col].dtype) for col in df.columns],
        "missing": [int(df[col].isna().sum()) for col in df.columns],
        "missing_share": [float(df[col].isna().mean()) for col in df.columns],
        "unique_values": [int(df[col].nunique(dropna=True)) for col in df.columns],
    }
).sort_values(["missing_share", "column"], ascending=[False, True])

overview = pd.DataFrame(
    [
        {"metric": "rows", "value": len(df)},
        {"metric": "columns", "value": df.shape[1]},
        {"metric": "first_article_date", "value": df["published_at"].min().date()},
        {"metric": "last_article_date", "value": df["published_at"].max().date()},
        {"metric": "months_covered", "value": df["month"].nunique()},
        {"metric": "unique_sources", "value": df["source"].replace("", np.nan).nunique()},
        {"metric": "unique_urls", "value": df["url"].replace("", np.nan).nunique()},
    ]
)

display(overview)
display(schema_summary.head(20))

## 4. Time Coverage and Article Volume

Monthly article volume is important because the modeling table aggregates news signals by month.

In [ ]:
START_DATE = "2020-01-01"
END_DATE = "2026-12-31"

period_df = df.loc[df["published_at"].between(START_DATE, END_DATE)].copy()

monthly_volume = (
    period_df.groupby("month", as_index=False)
    .agg(
        article_count=("title", "count"),
        unique_sources=("source", "nunique"),
        median_words=("estimated_words", "median"),
    )
    .sort_values("month")
)

time_summary = pd.DataFrame(
    [
        {"metric": "articles_in_2020_2026", "value": len(period_df)},
        {"metric": "months_in_2020_2026", "value": monthly_volume["month"].nunique()},
        {"metric": "avg_articles_per_month", "value": monthly_volume["article_count"].mean()},
        {"metric": "median_articles_per_month", "value": monthly_volume["article_count"].median()},
        {"metric": "min_monthly_articles", "value": monthly_volume["article_count"].min()},
        {"metric": "max_monthly_articles", "value": monthly_volume["article_count"].max()},
    ]
)

display(time_summary.round(2))
display(monthly_volume.head())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar(monthly_volume["month"], monthly_volume["article_count"], width=25, color="#2f6f73")
ax.set_title("Monthly Article Volume, 2020-2026")
ax.set_xlabel("Month")
ax.set_ylabel("Articles")
ax.grid(axis="y", alpha=0.25)
fig.autofmt_xdate()
plt.show()

yearly_volume = monthly_volume.assign(year=monthly_volume["month"].dt.year).groupby("year", as_index=False)["article_count"].sum()
display(yearly_volume)

## 5. Source Distribution

This checks whether the full loaded dataset is dominated by a few sources. Source concentration matters because one provider's editorial style can influence text length, sentiment, and topic mix.

In [ ]:
source_period_label = f"{df['published_at'].min().date()} to {df['published_at'].max().date()}"
print(f"Source distribution period: {source_period_label}")

source_counts = (
    df.assign(source=df["source"].replace("", "unknown"))
    .groupby("source", as_index=False)
    .agg(article_count=("title", "count"), median_words=("estimated_words", "median"))
    .sort_values("article_count", ascending=False)
)
source_counts["share"] = source_counts["article_count"] / source_counts["article_count"].sum()

display(source_counts.head(15).round({"share": 3, "median_words": 1}))

top_sources = source_counts.head(10).sort_values("article_count")
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top_sources["source"], top_sources["article_count"], color="#486581")
ax.set_title("Top News Sources")
ax.set_xlabel("Articles")
ax.grid(axis="x", alpha=0.25)
plt.show()

## 6. Missing Values and Duplicate URLs

This checks basic data quality for fields that matter in downstream feature engineering.

In [ ]:
key_columns = ["published_at", "month", "title", "source", "url", "eda_text", "word_count", "char_count"]
quality = pd.DataFrame(
    {
        "column": key_columns,
        "missing_or_blank": [int(df[col].isna().sum() + df[col].astype(str).str.strip().eq("").sum()) for col in key_columns],
        "missing_or_blank_share": [float((df[col].isna() | df[col].astype(str).str.strip().eq("")).mean()) for col in key_columns],
    }
)

non_empty_urls = df["url"].fillna("").astype(str).str.strip().ne("")
duplicate_url_rows = int(df.loc[non_empty_urls, "url"].duplicated().sum())
duplicate_title_source_rows = int(df.duplicated(subset=["title", "source", "published_at"]).sum())

duplicate_summary = pd.DataFrame(
    [
        {"metric": "duplicate_url_rows", "value": duplicate_url_rows},
        {"metric": "duplicate_title_source_date_rows", "value": duplicate_title_source_rows},
    ]
)

display(quality.round(3))
display(duplicate_summary)

if duplicate_url_rows:
    duplicate_examples = df.loc[non_empty_urls & df["url"].duplicated(keep=False), ["published_at", "source", "title", "url"]].sort_values("url")
    display(duplicate_examples.head(10))

## 7. Article Length Distribution

Text length is worth checking because very long articles can be liveblogs, daily summaries, or reports rather than standard news stories. Since article-length features are not used by the selected prediction model, this section is descriptive only.

In [ ]:
length_summary = period_df[["word_count", "char_count", "estimated_words"]].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
).T.round(1)
display(length_summary)

plot_words = period_df["estimated_words"].dropna()
plot_words = plot_words[plot_words > 0]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(plot_words, bins=40, color="#6c7a89", edgecolor="white")
if not plot_words.empty:
    ax.axvline(plot_words.median(), color="#2f6f73", linewidth=2, label=f"Median: {plot_words.median():.0f} words")
    ax.set_xscale("log")
ax.set_title("Article Length Distribution")
ax.set_xlabel("Estimated words per article, log scale")
ax.set_ylabel("Articles")
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False)
plt.show()

## 8. Long-Article Review

This is a review check, not an automatic modeling filter. It lists the longest articles and separately checks titles that look like liveblogs/daily summaries.

In [ ]:
title_lower = period_df["title"].fillna("").astype(str).str.lower()
liveblog_pattern = r"as it happened|business live|daily briefing|rolling coverage|latest updates"

review_df = period_df.copy()
review_df["liveblog_like_title"] = title_lower.str.contains(liveblog_pattern, regex=True, na=False)

longest_articles = review_df.sort_values("estimated_words", ascending=False)
liveblog_like_articles = review_df.loc[review_df["liveblog_like_title"]].sort_values("estimated_words", ascending=False)

review_summary = pd.DataFrame(
    [
        {"metric": "max_estimated_words", "value": review_df["estimated_words"].max()},
        {"metric": "p99_estimated_words", "value": review_df["estimated_words"].quantile(0.99)},
        {"metric": "mean_estimated_words", "value": review_df["estimated_words"].mean()},
        {"metric": "median_estimated_words", "value": review_df["estimated_words"].median()},
        {"metric": "liveblog_like_title_count", "value": len(liveblog_like_articles)},
        {"metric": "liveblog_like_title_share", "value": len(liveblog_like_articles) / len(review_df) if len(review_df) else np.nan},
    ]
)

display(review_summary.round(3))
display(longest_articles[["published_at", "source", "title", "estimated_words", "url"]].head(15))

if not liveblog_like_articles.empty:
    display(liveblog_like_articles[["published_at", "source", "title", "estimated_words", "url"]].head(15))